<a href="https://colab.research.google.com/github/almagomen/v2v/blob/monorepo-subtree/colab_launch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Video2Video – Despliegue en Colab

1. **Activa GPU**: *Runtime → Change runtime type → T4 GPU*
2. Ejecuta las celdas en orden.
3. Configura **Secrets** (panel izquierdo 🔑): `GOOGLE_API_KEY`, `LMNT_API_KEY`

## 1. Clonar repo e instalar dependencias

In [ ]:
# @title ⚠️ Un solo paso: clonar monorepo + construir OpenReel + levantar API (con túnel)

import os, time, threading, sys, warnings
from pathlib import Path
from google.colab import userdata, drive, output
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=SyntaxWarning)

# CONFIGURACIÓN
if not os.path.exists("/content/drive"): drive.mount("/content/drive")

GHP_TOKEN = userdata.get("GHP_GITHUB_TOKEN")
REPO_URL = f"https://{GHP_TOKEN}@github.com/almagomen/v2v.git"
REPO_PATH = Path("/content/v2v")
BRANCH_NAME = "monorepo-subtree"  # OpenReel subtree
BACKEND_PORT = 8000

# Clonar/actualizar
if not REPO_PATH.exists():
    print("📥 Clonando repositorio...")
    !git clone -q -b {BRANCH_NAME} {REPO_URL} {REPO_PATH}
else:
    print("🔄 Actualizando repositorio...")
    !git -C {REPO_PATH} fetch origin {BRANCH_NAME} -q
    !git -C {REPO_PATH} checkout {BRANCH_NAME} -q
    !git -C {REPO_PATH} reset --hard origin/{BRANCH_NAME} -q

if str(REPO_PATH) not in sys.path: sys.path.insert(0, str(REPO_PATH))

# Instalación de Dependencias
if 'deps_installed' not in globals():
    print("📦 Instalando dependencias...")
    !npm install -g pnpm -q
    !uv pip install git+https://github.com/jianfch/stable-ts.git git+https://github.com/pyannote/pyannote-audio.git --no-deps -q
    !uv pip install yt-dlp bs-roformer-infer tinydb sentencex torchcodec==0.7.0  'torch-audiomentations>=0.12.0'  'torchmetrics>=1.6.1' 'optuna>=4.2.0' -q
    !uv pip install openai-whisper 'pyannote-core>=6.0.1' 'pyannote-database>=6.1.1' 'pyannote-metrics>=4.0.0' 'pyannote-pipeline>=4.0.0' 'pyannoteai-sdk>=0.3.0' --no-deps -q
    !uv pip install 'opentelemetry-api>=1.34.0' 'opentelemetry-sdk>=1.34.0' 'opentelemetry-exporter-otlp>=1.34.0'  'asteroid-filterbanks>=0.4.0' 'lightning>=2.4' 'pytorch-metric-learning>=2.8.1'  -q

    print("🛠️ Cargando modelos...")
    from src.stt import get_whisper_model
    from src.diarization import get_diarization_model
    from src.download import get_roformer_model
    get_whisper_model()
    get_diarization_model()
    get_roformer_model()
    globals()['deps_installed'] = True

# Build OpenReel
OPENREEL_PATH = REPO_PATH / "openreel"
print("🏗️  Construyendo OpenReel (WASM + web)...")
!cd {OPENREEL_PATH} && pnpm install
!cd {OPENREEL_PATH/'packages'/'core'} && npm install
!cd {OPENREEL_PATH} && pnpm build

# Backend FastAPI
import uvicorn

class ControlledServer(uvicorn.Server):
    def install_signal_handlers(self): return

def run_backend():
    from api.main import app
    config = uvicorn.Config(app, host="0.0.0.0", port=BACKEND_PORT)
    server = ControlledServer(config=config)
    server.run()

print(f"🚀 Iniciando backend en 0.0.0.0:{BACKEND_PORT}...")
thread = threading.Thread(target=run_backend, daemon=True)
thread.start()

time.sleep(4)

# Túnel Nativo
print("\n✨ ENTORNO LISTO ✨")
output.serve_kernel_port_as_window(BACKEND_PORT, path="/", skip_warning=True)
